In [0]:
file_location = "/Volumes/workspace/default/cis5560_dat/baseball_salaries_2003v2.txt"

df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ":") \
    .load(file_location)

display(df)

In [0]:
df = df.filter(~df["_c0"].startswith("#"))

In [0]:
df = df.withColumnRenamed("_c0", "Team") \
       .withColumnRenamed("_c1", "Player") \
       .withColumnRenamed("_c2", "Salary") \
       .withColumnRenamed("_c3", "Position")

In [0]:
from pyspark.sql.functions import trim, regexp_replace, col

df = df.withColumn("Team", trim(col("Team"))) \
       .withColumn("Player", trim(regexp_replace(col("Player"), '"', ''))) \
       .withColumn("Salary", trim(col("Salary")).cast("int")) \
       .withColumn("Position", trim(col("Position")))

In [0]:
display(df)

In [0]:
print("Number of rows:", df.count())
print("Columns:", df.columns)
df.printSchema()

In [0]:
df.createOrReplaceTempView("baseball_salaries")

In [0]:
%sql

SELECT *
FROM baseball_salaries
ORDER BY Salary DESC
LIMIT 5;

In [0]:
from pyspark.sql.functions import col

top5_df = df.orderBy(col("Salary").desc()).limit(5)

display(top5_df)

In [0]:
%sql

SELECT *
FROM baseball_salaries
WHERE Position = 'First Baseman'
ORDER BY Salary DESC
LIMIT 1;

In [0]:
highest_first_baseman = df.filter(
    col("Position") == "First Baseman"
).orderBy(
    col("Salary").desc()
).limit(1)

display(highest_first_baseman)

In [0]:
%sql

SELECT
    Position,
    AVG(Salary) AS Average_Salary
FROM baseball_salaries
GROUP BY Position
ORDER BY Position;

In [0]:
from pyspark.sql.functions import avg

average_salary = df.groupBy("Position") \
    .agg(avg("Salary").alias("Average_Salary")) \
    .orderBy("Position")

display(average_salary)